# Poker Solver — GPU Benchmark (multi-street CFR)

Confirms whether the batched multi-street solver hits **minutes/board** on a GPU.
On CPU (`docs/multistreet_spike.md`) the river solve is ~hours/board at full ranges;
the showdown is a batched matrix multiply, so a GPU should cut it sharply.

**Before running:** `Runtime -> Change runtime type -> GPU -> Save`.

**Upload the code:** click the **folder icon** on the left sidebar, then the
**Upload to session storage** button, and pick `poker_solver_gpu.zip`. (Do this
instead of an in-cell upload widget — the widget can fail with `Failed to fetch`.)

Then run the cells top to bottom.

In [ ]:
# 1) Confirm a GPU is attached
!nvidia-smi -L || echo 'NO GPU — set Runtime -> Change runtime type -> GPU'

In [ ]:
# 2) Unpack the code.
#
# UPLOAD FIRST (reliable, no widget): click the FOLDER icon on the left sidebar
# -> the "Upload to session storage" button (page-with-up-arrow) -> pick
# poker_solver_gpu.zip. Wait for its progress ring to finish, THEN run this cell.
#
# (This avoids files.upload(), which can throw 'Failed to fetch' in some browsers.)
import zipfile, os, glob
hits = glob.glob('/content/poker_solver_gpu.zip') or glob.glob('/content/*.zip')
if not hits:
    raise FileNotFoundError(
        "No zip in /content. Use the left sidebar folder icon -> Upload, pick "
        "poker_solver_gpu.zip, wait for it to finish, then re-run this cell.")
with zipfile.ZipFile(hits[0]) as z:
    z.extractall('poker')
print('unpacked', hits[0], '->', sorted(os.listdir('poker')))

In [ ]:
# 3) Verify CuPy sees the GPU (Colab GPU runtimes usually ship CuPy preinstalled;
#    this self-heals with a pip install if the import fails).
try:
    import cupy as cp
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"])
    import cupy as cp
name = cp.cuda.runtime.getDeviceProperties(0)['name']
if isinstance(name, bytes):
    name = name.decode()
print('cupy', cp.__version__, '| device:', name)

In [ ]:
# 4) Correctness first: the GPU backend must match the CPU backend.
import numpy as np, sys
sys.path.insert(0, 'poker/src')
from pokertrainer.cards import parse_cards, parse_hand
from pokertrainer.solver.batched_gpu import BatchedGPUCFR
flop = parse_cards('As7h2d')
oop = [parse_hand(h) for h in ['AhAc','KsKc','7s7c','AhKh','Ts9s']]
ip  = [parse_hand(h) for h in ['AhQh','KsKh','JsJh','AhTh','Tc9c']]
wo, wi = np.ones(5), np.ones(5)
g = BatchedGPUCFR(flop, oop, ip, wo, wi, 5.5, 0.66, streets=2, backend='cupy').run(200)
c = BatchedGPUCFR(flop, oop, ip, wo, wi, 5.5, 0.66, streets=2, backend='numpy').run(200)
diff = abs(g['root_ev_oop_bb'] - c['root_ev_oop_bb'])
print('GPU EV %.6f%%  |  CPU EV %.6f%%  |  diff %.2e' %
      (g['root_ev_pct_pot'], c['root_ev_pct_pot'], diff))
print('PASS (GPU==CPU)' if diff < 1e-6 else 'CHECK — diff larger than float rounding')

## 5) The benchmark — GPU vs CPU

`streets=3` is the full **river** solve. Columns: combos/side, cache-build time,
and steady **ms/iter**. Convergence needs ~500–1000 iterations, so
`ms/iter × 600 / 1000` ≈ **seconds/board**, and `× 12 / 60` ≈ **minutes for a
12-board library**.

In [ ]:
# 6) GPU in float64 (exact) — the baseline
%env POKER_BACKEND=cupy
%env POKER_DTYPE=float64
!cd poker && PYTHONPATH=src python bench/gpu_bench.py 3 80 160 250

In [ ]:
# 7) CPU baseline for comparison (smaller sizes — CPU is slow at streets=3)
%env POKER_BACKEND=numpy
%env POKER_DTYPE=float64
!cd poker && PYTHONPATH=src python bench/gpu_bench.py 3 40 80

In [ ]:
## Reading the result

Compare **ms/iter** at ~250 combos across the three runs:
- **cell 6** — GPU float64 (exact)
- **cell 6b** — GPU float32 (≈1e-6 of pot less accurate; usually much faster on a T4)
- **cell 7** — CPU float64 (baseline)

Convert: `ms/iter × 600 / 1000` ≈ **seconds/board**, `× 12 / 60` ≈ **minutes for a 12-board library**.

- **Go (minutes/board):** the fastest GPU row at ~250 combos lands in the tens-to-low-hundreds
  of ms → seconds-to-minutes/board → a full-range multi-street library is practical, fully
  permissive (CuPy is BSD/MIT-style), zero copyleft. A datacenter GPU (A100/H100) adds another
  ~10–30× on top of a T4.
- **Marginal:** still seconds/iter even in float32 → needs isomorphism / range abstraction, or
  a beefier GPU, to be practical at scale.

Paste all three tables back and we'll lock the decision.

## Reading the result

- **Go (minutes/board):** GPU ms/iter at ~250 combos is in the tens-to-low-hundreds
  of ms → seconds-to-minutes per board → a full-range multi-street library is
  practical, fully permissive (CuPy is BSD/MIT-style), zero copyleft.
- **No-go:** GPU barely beats CPU (unlikely for a batched matmul) → revisit
  isomorphism / range abstraction, or reconsider a licensed backend.

Paste the two tables back and we'll lock the decision.